In [1]:
def float_to_int_in_range(value, in_min=0.0, in_max=1.0, out_min=1, out_max=10):
    # Clamp the value to ensure it stays within the input range
    value = max(in_min, min(value, in_max))
    # Normalize the value to [0, 1]
    normalized = (value - in_min) / (in_max - in_min)
    # Scale and shift to the desired output range
    result = int(round(normalized * (out_max - out_min) + out_min))
    return result

# Test with some example values
test_values = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
for v in test_values:
    print(f"{v} -> {float_to_int_in_range(v)}")


0.0 -> 1
0.1 -> 2
0.25 -> 3
0.5 -> 6
0.75 -> 8
0.9 -> 9
1.0 -> 10


In [2]:
import re

def float_to_int_in_range(value, in_min=0.0, in_max=1.0, out_min=1, out_max=10):
    # Clamp the value
    value = max(in_min, min(value, in_max))
    # Normalize to [0, 1]
    normalized = (value - in_min) / (in_max - in_min)
    # Scale to desired range and round
    result = int(round(normalized * (out_max - out_min) + out_min))
    return result

def extract_float_array_from_header(header_file):
    """
    Extracts the C array of floats from the given header file.
    Assumes the array is defined like: const float model_weights[] = { ... };
    """
    with open(header_file, 'r') as f:
        content = f.read()
    
    # Regex to capture everything between the first '{' and the matching '}'
    pattern = re.compile(r'\{([^}]+)\}')
    match = pattern.search(content)
    if not match:
        raise ValueError("Could not find a C array in the header file.")
    
    numbers_str = match.group(1)
    # Split by comma and filter out empty strings
    numbers = [s.strip() for s in numbers_str.split(',') if s.strip()]
    # Convert to floats
    float_values = [float(num) for num in numbers]
    return float_values

def convert_float_array_to_int_array(float_array, in_min=0.0, in_max=1.0, out_min=1, out_max=10):
    """
    Converts a list of float values to integer values using float_to_int_in_range.
    """
    return [float_to_int_in_range(val, in_min, in_max, out_min, out_max) for val in float_array]

def write_int_array_to_header(int_array, output_header_file, array_name="model_weights"):
    """
    Writes the integer array to a new header file in C syntax.
    """
    # Create a C array string from the integer values
    array_str = ", ".join(str(val) for val in int_array)
    header_content = f"const int {array_name}[] = {{ {array_str} }};\n"
    header_content += f"const unsigned int {array_name}_len = sizeof({array_name});\n"
    with open(output_header_file, "w") as f:
        f.write(header_content)
    print(f"New header file saved as '{output_header_file}'.")

# File paths (change these as needed)
input_header_file = "mnist_model_weights.h"
output_header_file = "mnist_model_weights_int.h"

# Extract float array from the original header file
float_array = extract_float_array_from_header(input_header_file)
print("Extracted float array with", len(float_array), "elements.")

# Convert float values to int values (range 1 to 10)
int_array = convert_float_array_to_int_array(float_array, in_min=0.0, in_max=1.0, out_min=1, out_max=10)
print("Converted values (first 20):", int_array[:20])

# Write the new integer array to a header file
write_int_array_to_header(int_array, output_header_file)


Extracted float array with 25450 elements.
Converted values (first 20): [1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
New header file saved as 'mnist_model_weights_int.h'.


In [3]:
import re

def float_to_int8(value, in_min=-1.0, in_max=1.0):
    """
    Convert a float value to an int8 using linear quantization.
    Assumes that value is in [in_min, in_max] and maps it to [-128, 127].
    """
    # Clamp value within the input range
    value = max(in_min, min(value, in_max))
    # Normalize to [0, 1]
    normalized = (value - in_min) / (in_max - in_min)
    # Map to int8 range: 0->-128, 1->127
    int_val = int(round(normalized * 255)) - 128
    return int_val

def extract_float_array_from_header(header_file):
    """
    Extracts the C array of floats from the given header file.
    Assumes the array is defined like: const float model_weights[] = { ... };
    """
    with open(header_file, 'r') as f:
        content = f.read()
    
    # Regex to capture everything between the first '{' and the matching '}'
    pattern = re.compile(r'\{([^}]+)\}')
    match = pattern.search(content)
    if not match:
        raise ValueError("Could not find a C array in the header file.")
    
    numbers_str = match.group(1)
    # Split by comma, remove empty strings, and convert to floats
    float_values = [float(s.strip()) for s in numbers_str.split(',') if s.strip()]
    return float_values

def convert_float_array_to_int8_array(float_array, in_min=-1.0, in_max=1.0):
    """
    Converts a list of float values to int8 values using float_to_int8.
    """
    return [float_to_int8(val, in_min, in_max) for val in float_array]

def write_int8_array_to_header(int8_array, output_header_file, array_name="model_weights"):
    """
    Writes the int8 array to a new header file in C syntax using int8_t.
    """
    # Create a C array string from the integer values
    array_str = ", ".join(str(val) for val in int8_array)
    header_content = (
        "#include <avr/pgmspace.h>\n\n"
        f"const int8_t {array_name}[] PROGMEM = {{ {array_str} }};\n"
        f"const unsigned int {array_name}_len = sizeof({array_name});\n"
    )
    with open(output_header_file, "w") as f:
        f.write(header_content)
    print(f"New header file saved as '{output_header_file}'.")

# File paths (adjust as needed)
input_header_file = "mnist_model_weights.h"         # Original header file with float array
output_header_file = "mnist_model_weights_int.h"      # New header file with int8 array

# Step 1: Extract float array from the original header file
float_array = extract_float_array_from_header(input_header_file)
print("Extracted float array with", len(float_array), "elements.")

# Step 2: Convert float array to int8 array using the provided range (default: [-1, 1])
int8_array = convert_float_array_to_int8_array(float_array, in_min=-1.0, in_max=1.0)
print("Converted values (first 20):", int8_array[:20])

# Step 3: Write the new int8 array to a header file
write_int8_array_to_header(int8_array, output_header_file)


Extracted float array with 25450 elements.
Converted values (first 20): [-11, 1, 10, -2, 10, -4, -8, -8, 0, -7, -7, -8, -5, -3, 2, 4, -10, -10, 0, -4]
New header file saved as 'mnist_model_weights_int.h'.
